# Chapter 27: Front End vs Back End

<a href="../lite/lab/index.html?path=ch27_frontend_backend.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

Every SLAM system has two halves that barely speak to each other. The **front end** looks at
raw sensor data and says "I saw landmark 7 at bearing 45 degrees." The **back end** takes those
reports and solves a giant math problem. When SLAM fails, the first question is always:
did the front end report garbage, or did the back end optimize wrong?

Understanding this separation is essential for debugging and designing SLAM systems.

## 27.1 Perception vs Estimation

The **front end** (perception) handles:
- Feature detection and tracking
- Data association (which feature matches which)
- Outlier rejection

The **back end** (estimation) handles:
- State estimation (poses and map)
- Optimization (least squares, graph SLAM)
- Uncertainty quantification

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_poses = 20
n_landmarks = 6
front_end_error_rate = 0.1    # fraction of wrong associations  (try 0, 0.1, 0.3)
back_end_iterations = 5       # optimizer iterations
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Ground truth
angles = np.linspace(0, 2*np.pi, n_poses, endpoint=False)
true_poses = 5 * np.column_stack([np.cos(angles), np.sin(angles)])
landmarks = np.random.uniform(-7, 7, (n_landmarks, 2))

# Front-end: generate observations (some with errors)
observations = []
for i in range(n_poses):
    for j in range(n_landmarks):
        if np.linalg.norm(true_poses[i] - landmarks[j]) < 8:
            z = landmarks[j] - true_poses[i] + np.random.normal(0, 0.3, 2)
            assoc = j
            if np.random.random() < front_end_error_rate:
                assoc = (j + 1) % n_landmarks  # wrong association!
            observations.append((i, assoc, z))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.set_title("Front End: raw observations", fontsize=13)
ax.plot(true_poses[:, 0], true_poses[:, 1], 'steelblue', lw=2, marker='o', ms=4)
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='tomato', s=100, marker='^', zorder=5)
ax.set_aspect('equal')

ax = axes[1]
ax.set_title(f"Back End: estimation ({front_end_error_rate:.0%} front end errors)", fontsize=13)
ax.plot(true_poses[:, 0], true_poses[:, 1], 'k--', lw=1, alpha=0.5, label='ground truth')
# Simulate noisy estimation
noise_level = 0.2 + 2 * front_end_error_rate
est_poses = true_poses + np.random.normal(0, noise_level, true_poses.shape)
ax.plot(est_poses[:, 0], est_poses[:, 1], 'steelblue', lw=2, marker='o', ms=4, label='estimated')
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='tomato', s=100, marker='^', zorder=5)
ax.set_aspect('equal'); ax.legend()

plt.suptitle("Front end errors → back end corruption", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Total observations: {len(observations)}")
print(f"Front end error rate: {front_end_error_rate:.0%}")
print(f"Approximate wrong associations: {int(len(observations) * front_end_error_rate)}")

## 27.2 Responsibilities

| Component | Input | Output | Fails when... |
|-----------|-------|--------|---------------|
| **Front end** | Raw sensor data | Feature tracks, associations | Perceptual aliasing, textureless scenes |
| **Back end** | Observations + associations | Optimized poses + map | Bad associations, degenerate geometry |

## 27.3 Debugging

When SLAM fails, diagnose systematically:
1. **Check the front end first**: are features being tracked correctly? Are associations right?
2. **Then check the back end**: is the optimizer converging? Is the initial guess reasonable?
3. **Common front end failures**: wrong loop closures, lost tracking, outlier contamination
4. **Common back end failures**: bad initial guess (local minimum), gauge freedom issues

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# Simulate two failure modes
failure_mode = "frontend"     # try "frontend" or "backend"
# ──────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(true_poses[:, 0], true_poses[:, 1], 'k--', lw=1, alpha=0.5, label='ground truth')

if failure_mode == "frontend":
    # Front end error: one wrong loop closure
    bad_poses = true_poses.copy()
    bad_poses[10:] += np.array([1.5, -1.0])  # drift from wrong association
    ax.plot(bad_poses[:, 0], bad_poses[:, 1], 'tomato', lw=2, marker='o', ms=4, label='front end error')
    ax.annotate("wrong association\nhere", xy=bad_poses[10], fontsize=11, color='tomato',
                xytext=(bad_poses[10, 0]+1, bad_poses[10, 1]+1.5),
                arrowprops=dict(arrowstyle='->', color='tomato'))
else:
    # Back end error: optimizer stuck in local minimum
    bad_poses = true_poses + 0.8 * np.sin(np.arange(n_poses))[:, None]
    ax.plot(bad_poses[:, 0], bad_poses[:, 1], 'tomato', lw=2, marker='o', ms=4, label='back end error (local min)')

ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=100, marker='^', zorder=5)
ax.set_aspect('equal'); ax.legend()
ax.set_title(f"Failure mode: {failure_mode}", fontsize=13)
plt.tight_layout()
plt.show()

**Key observations:**
- A clean separation between front end and back end makes debugging much easier.
- Front end errors (wrong data) are harder to fix than back end errors (wrong math).
- Always validate the front end before blaming the optimizer.

---

## Exercises

### Exercise 27.1
Given a set of observations, inject 20% wrong associations. Run a simple least squares
estimator. Plot the result and compare with the 0% error case. How many wrong associations
can your system tolerate before the map becomes unusable?

In [ ]:
# Your code here